# Лабораторная работа 2

Вариант: искусство. X - width_mm, G - десятилетие по acquired

## Задание 1. Подготовка

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

## Задание 2. Загрузка данных

In [ ]:
df = pd.read_csv('records.csv')
df.head(5)

In [ ]:
print(df.dtypes)
print(df['width_mm'].isna().sum())

In [ ]:
decades = []
for a in df['acquired']:
    if pd.isna(a):
        decades.append('Неизвестно')
    else:
        decades.append(str(int(a // 10 * 10)))

df['decade'] = decades

width_mm это float, то есть числа. Пустые ячейки стали пропусками NaN, их 381.

Для G сделала столбец decade: например 1995 -> 1990. Если acquired пустой, то пишу "Неизвестно".

## Задание 3. Пропуски

In [ ]:
df3 = df.copy()

removed = df3[df3['width_mm'].notna()].sample(frac=0.1, random_state=SEED)
removed_index = removed.index
removed_values = removed['width_mm']

df3.loc[removed_index, 'width_mm'] = np.nan
print(df3['width_mm'].isna().sum())

In [ ]:
mean = df3['width_mm'].mean()
median = df3['width_mm'].median()

df_mean = df3.copy()
df_mean['width_mm'] = df_mean['width_mm'].fillna(mean)

df_median = df3.copy()
df_median['width_mm'] = df_median['width_mm'].fillna(median)

df_dec = df3.copy()
for d in df_dec['decade'].unique():
    med = df_dec[df_dec['decade'] == d]['width_mm'].median()
    df_dec.loc[(df_dec['decade'] == d) & (df_dec['width_mm'].isna()), 'width_mm'] = med

In [ ]:
diff1 = df_mean.loc[removed_index, 'width_mm'] - removed_values
diff2 = df_median.loc[removed_index, 'width_mm'] - removed_values
diff3 = df_dec.loc[removed_index, 'width_mm'] - removed_values

print('среднее: MAE =', np.mean(np.abs(diff1)), 'MSE =', np.mean(diff1 ** 2))
print('медиана: MAE =', np.mean(np.abs(diff2)), 'MSE =', np.mean(diff2 ** 2))
print('медиана по десятилетию: MAE =', np.mean(np.abs(diff3)), 'MSE =', np.mean(diff3 ** 2))

Лучше всего заполнять медианой по десятилетию, у неё меньше и MAE, и MSE. Ширина работ в разные десятилетия разная (в 1850-х в основном маленькие рисунки, а в 2000-х работы больше).

Медиана лучше среднего по MAE, а среднее лучше по MSE. В данных есть очень широкие работы (до 8920 мм), они тянут среднее вверх (326 против медианы 175). В MSE ошибки возводятся в квадрат, поэтому большие ошибки сильно влияют, а в MAE все ошибки считаются одинаково. Поэтому метрики получаются разные.

## Задание 4. Дубликаты

In [ ]:
df4 = df.copy()
copies = df4.sample(frac=0.05, random_state=SEED)
df_dup = pd.concat([df4, copies])

d1 = df_dup.copy().drop_duplicates()
d2 = df_dup.copy().drop_duplicates(subset='id')
d3 = df_dup.copy().drop_duplicates(subset='title')

print('до копий:', len(df4), df4['width_mm'].mean())
print('после копий:', len(df_dup), df_dup['width_mm'].mean())
print('по всем столбцам:', len(d1), d1['width_mm'].mean())
print('по id:', len(d2), d2['width_mm'].mean())
print('по title:', len(d3), d3['width_mm'].mean())

In [ ]:
title_id_counts = d1.groupby('title')['id'].nunique()
different_ids = title_id_counts[title_id_counts > 1].index
same_title = d1[d1['title'].isin(different_ids)].sort_values('title')
print(len(same_title))
same_title[['id', 'title', 'artist', 'width_mm']].head(10)

По всем столбцам и по id удалились только добавленные копии, осталось 8000 строк и среднее как было.

По title удалилось намного больше (осталось 6038), потому что у разных работ бывает одинаковое название, например "[title not known]" или наброски Тёрнера с одинаковыми названиями. У них разные id, то есть это разные работы, и удалять их неправильно.

## Задание 5. Числовые характеристики

In [ ]:
values = df['width_mm'].dropna().tolist()
n = len(values)
values.sort()

razmah = values[-1] - values[0]

s = 0
for v in values:
    s += v
sr = s / n

if n % 2 == 1:
    med = values[n // 2]
else:
    med = (values[n // 2 - 1] + values[n // 2]) / 2

s2 = 0
for v in values:
    s2 += (v - sr) ** 2
disp = s2 / n

std = disp ** 0.5

print(razmah, sr, med, disp, std)

In [ ]:
x = df['width_mm'].dropna()
print(np.max(x) - np.min(x), np.mean(x), np.median(x), np.var(x), np.std(x))
print(x.max() - x.min(), x.mean(), x.median(), x.var(), x.std())

С NumPy всё совпало. В pandas дисперсия и std немного другие, потому что pandas делит на n-1, а не на n.

## Задание 6. Фильтрация

In [ ]:
t1 = df[(df['decade'].isin(['1970', '1990'])) & (df['width_mm'] > 500)]
t2 = df[(df['decade'].isin(['1970', '1990'])) | (df['width_mm'] > 500)]
print(len(t1), len(t2))

In [ ]:
only2 = t2.drop(t1.index)
print(len(only2))
only2.head(10)

Выбрала десятилетия 1970 и 1990 и границу 500 мм. Во второй таблице строк намного больше, потому что там хватает одного условия: туда попали все работы из 1970 и 1990 (даже узкие) и широкие работы из других десятилетий.

## Задание 7. Порядок операций

In [ ]:
a = t1.sort_values('width_mm', ascending=False).head(5)
a

In [ ]:
b = df.sort_values('width_mm', ascending=False).head(5)
b = b[(b['decade'].isin(['1970', '1990'])) & (b['width_mm'] > 500)]
b

Получилось по-разному. В первом случае 5 строк, а во втором только 2, потому что среди 5 самых широких работ всей таблицы три из 2000-х и 2010-х, а из 1970 и 1990 только две. Так что результат не всегда одинаковый, он совпадёт только если топ-5 всей таблицы подходит под условие.

## Задание 8. Группировка

In [ ]:
g = df.groupby('decade')

stats = pd.DataFrame()
stats['count'] = g.size()
stats['share'] = stats['count'] / len(df)
stats['missing'] = stats['count'] - g['width_mm'].count()
stats['mean'] = g['width_mm'].mean()
stats['median'] = g['width_mm'].median()
stats = stats.sort_values('count', ascending=False)
stats

In [ ]:
c = stats['count'].sort_values()
plt.barh(c.index, c.values)
plt.xlabel('количество')
plt.ylabel('десятилетие')
plt.tight_layout()
plt.show()

Больше всего работ поступило в 1850-х, это больше половины таблицы (почти все в 1856 году). Потом идут 1970, 1990, 1980 и 2000. Меньше всего в 1820, 1830, 1860 и группе "Неизвестно".

Больше всего пропусков ширины в 1880-х (108 из 121).

## Задание 9. Объект и его группа

In [ ]:
df['group_count'] = df['decade'].map(stats['count'])
df['group_mean'] = df['decade'].map(stats['mean'])
print(len(df))

df['delta'] = df['width_mm'] - df['group_mean']
df.head(5)

In [ ]:
part = df[df['width_mm'].notna()].head(100)

plt.scatter(range(1, len(part) + 1), part['delta'])
plt.axhline(0, color='red')
plt.xlabel('номер')
plt.ylabel('delta')
plt.show()

In [ ]:
part2 = part.sort_values('delta')

plt.plot(range(1, len(part2) + 1), part2['delta'])
plt.xlabel('номер')
plt.ylabel('delta')
plt.show()

В таблице осталось 8000 строк, потому что новые столбцы добавлены к исходному DataFrame, а строки не удалялись. `group_count` одинаков для всех работ одного десятилетия, `group_mean` показывает среднюю ширину этого десятилетия, а `delta` показывает отклонение конкретной работы от среднего. На первом графике точка выше нуля означает ширину выше среднего группы. После сортировки `delta` линия не убывает, потому что значения расположены от меньшего к большему.

## Задание 10. Квантили и boxplot

In [ ]:
df['width_mm'].quantile([0.25, 0.5, 0.75])

In [ ]:
top = stats.head(5).index

data = []
for d in top:
    data.append(df[df['decade'] == d]['width_mm'].dropna())

plt.boxplot(data)
plt.xticks([1, 2, 3, 4, 5], top)
plt.xlabel('десятилетие')
plt.ylabel('ширина, мм')
plt.show()

Медиана выше всего у 2000 и 1970, ниже всего у 1850. Сильнее всего разброс у 2000, у неё самый большой ящик, а у 1850 ящик совсем маленький, там почти все работы небольшие.